In [1]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
# output_file = "llama3.1_result.jsonl"
output_file = "llama8b_r1_result.jsonl"

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

def load_model(model_name, device="auto"):
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = LLM(
        model=model_name,
        tensor_parallel_size=8,  # set >1 if using multiple GPUs
        dtype="bfloat16",      
        gpu_memory_utilization=0.9
)
    return model, tokenizer

model, tokenizer = load_model(model_name)

INFO 08-24 01:58:02 [__init__.py:239] Automatically detected platform cuda.
INFO 08-24 01:58:12 [config.py:600] This model supports multiple tasks: {'reward', 'generate', 'score', 'classify', 'embed'}. Defaulting to 'generate'.
INFO 08-24 01:58:12 [config.py:1600] Defaulting to use mp for distributed inference
INFO 08-24 01:58:12 [config.py:1780] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-24 01:58:14 [core.py:61] Initializing a V1 LLM engine (v0.8.3) with config: model='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(VllmWorker rank=1 pid=1948528) INFO 08-24 01:58:21 [loader.py:447] Loading weights took 1.29 seconds
(VllmWorker rank=5 pid=1948635) INFO 08-24 01:58:21 [loader.py:447] Loading weights took 1.24 seconds
(VllmWorker rank=4 pid=1948608) INFO 08-24 01:58:21 [loader.py:447] Loading weights took 1.20 seconds
(VllmWorker rank=3 pid=1948580) INFO 08-24 01:58:21 [loader.py:447] Loading weights took 1.20 seconds
(VllmWorker rank=2 pid=1948555) INFO 08-24 01:58:22 [loader.py:447] Loading weights took 1.21 seconds
(VllmWorker rank=1 pid=1948528) INFO 08-24 01:58:22 [gpu_model_runner.py:1273] Model loading took 1.9029 GiB and 1.698841 seconds
(VllmWorker rank=7 pid=1948691) INFO 08-24 01:58:22 [loader.py:447] Loading weights took 1.09 seconds
(VllmWorker rank=5 pid=1948635) INFO 08-24 01:58:22 [gpu_model_runner.py:1273] Model loading took 1.9029 GiB and 1.749335 seconds
(VllmWorker rank=0 pid=1948505) INFO 08-24 01:58:22 [loader.py:447] Loading weights took 1.25 seconds
(VllmWorker rank=4 pid=194

In [3]:
# Text Generation
if "Llama" in model_name:
    BOS = 128000
    USER = 128011
    ASSISTANT = 128012
    NEWLINE = 198
    THINK_START = 128013
    THINK_END = 128014
    EOS = 128001
elif "Qwen" in model_name:
    BOS = 151646
    USER = 151644
    ASSISTANT = 151645
    NEWLINE = 198
    THINK_START = 151648
    THINK_END = 151649
    EOS = 151643
else:
    raise ValueError(f"Unknown tokens for model {model_name}")

In [4]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

def pprint(text):
    """Pretty print the model's generated text using rich."""
    console = Console(width=100)
    
    
    # Create markdown and display in a panel
    #md = Markdown(text.strip())
    console.print(Panel(text, border_style="blue"))


In [5]:
# load math 500 dataset: HuggingFaceH4/MATH-500
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/MATH-500")

if "R1" in model_name:
    def prompt_from_example(example, tokenizer):
        user_message = example['problem']
        math_suffix = " Please reason step by step, and put your final answer within \\boxed{}."
        toks = [BOS] + [USER] + tokenizer.encode(user_message+math_suffix, add_special_tokens=False) + [ASSISTANT] + [THINK_START] + [NEWLINE]
        #toks = [BOS] + tokenizer.encode(user_message+math_suffix, add_special_tokens=False) + [THINK_START] + [NEWLINE]
        return toks, tokenizer.decode(toks, skip_special_tokens=False)
else:
    def prompt_from_example(example, tokenizer):
        user_message = example['problem']
        math_suffix = " Please reason step by step, and put your final answer within \\boxed{}."
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_message + math_suffix},
        ]
        toks = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        return toks, tokenizer.decode(toks, skip_special_tokens=False)

In [6]:
toks, prompt = prompt_from_example(dataset['test'][0], tokenizer)
answer = dataset['test'][0]['answer']

In [21]:
tokenizer

LlamaTokenizerFast(name_or_path='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', vocab_size=128000, model_max_length=16384, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>', 'pad_token': '<｜end▁of▁sentence｜>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	128000: AddedToken("<｜begin▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<｜end▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=T

In [7]:
# generate a continuation of prompt using vllm

if "R1" in model_name:
    sampling_params = SamplingParams(
        temperature=0.6,
        top_p=0.95,
        max_tokens=32768
    )
else:
    sampling_params = SamplingParams(
        temperature=0.6,
        top_p=0.95,
        max_tokens=15000
    )

In [8]:
out = model.generate([prompt], sampling_params)
generated_text = out[0].outputs[0].text
print(generated_text)

Processed prompts: 100%|██████████| 1/1 [00:10<00:00, 10.25s/it, est. speed input: 6.63 toks/s, output: 84.20 toks/s]

Okay, so I need to convert the rectangular coordinates (0, 3) to polar coordinates. Hmm, let me remember how to do this. I think polar coordinates are represented as (r, θ), right? Where r is the distance from the origin and θ is the angle made with the positive x-axis. 

First, I recall that the formulas to convert from rectangular (x, y) to polar (r, θ) are:

r = sqrt(x² + y²)
θ = arctan(y / x)

But wait, I should be careful with the quadrant because depending on the signs of x and y, θ might need to be adjusted. In this case, the point is (0, 3). So, x is 0 and y is 3. Let me plug these into the formulas.

Calculating r first. So, r is the square root of (0 squared plus 3 squared). That would be sqrt(0 + 9) which is sqrt(9). The square root of 9 is 3. So, r is 3. That seems straightforward.

Now, calculating θ. The formula is arctan(y / x). Plugging in the values, that's arctan(3 / 0). Wait a second, dividing by zero is undefined. Hmm, what does that mean? I remember that arctan(∞) 

In [18]:
import re

import sys
sys.path.append("/disk/u/troitskiid/projects/r1helpers")
from src.r1helpers.math500.grader import grade_answer

if "R1" in model_name:
    def parse_answer(generated_text):
        matches = re.search("</think>", generated_text)
        if matches is None:
            return ""
        generated_answer = generated_text[matches.end():]
        # in generated answer select the content of \boxed{}
        matches = re.search("\\\\boxed{", generated_answer)
        if matches is None:
            return ""
        generated_answer = generated_answer[matches.end():]
        # search all the way to the end of the string   }
        reversed = generated_answer[::-1]
        matches = re.search("}", reversed)
        if matches is None:
            return ""
        generated_answer = generated_answer[:len(generated_answer) - matches.start()]
        return generated_answer
else:
    def parse_answer(generated_text):
        end_idx = generated_text.find("<|start_header_id|>assistant<|end_header_id|>")
        generated_answer = generated_text[end_idx:]
        matches = re.search("\\\\boxed{", generated_answer)
        if matches is None:
            return ""
        generated_answer = generated_answer[matches.end():]
        # search all the way to the end of the string   }
        reversed = generated_answer[::-1]
        matches = re.search("}", reversed)
        if matches is None:
            return ""
        generated_answer = generated_answer[:len(generated_answer) - matches.start()]
        return generated_answer

parsed_answer = parse_answer(generated_text)
print(parsed_answer)
print(parsed_answer, answer)
grade_answer(parsed_answer, answer)

(3, \frac{\pi}{2})}
(3, \frac{\pi}{2})} \left( 3, \frac{\pi}{2} \right)


True

In [ ]:
# evaluate question by question and store results in both dataframe and jsonl file
import pandas as pd
import json
from tqdm import tqdm

df = pd.DataFrame(columns=['problem', 'answer', 'generated_answer', 'correct'])
n_correct = 0
for idx, example in tqdm(enumerate(dataset['test'])):
    toks, prompt = prompt_from_example(example, tokenizer)
    out = model.generate([prompt], sampling_params)
    generated_text = out[0].outputs[0].text 
    generated_answer = parse_answer(generated_text)
    example['parsed_answer'] = generated_answer
    example['correct'] = grade_answer(generated_answer, example['answer'])
    example['generated_text'] = generated_text
    n_correct += int(example['correct'])
    print(n_correct / (idx + 1))

    # add to both dataframe and jsonl file
    df = pd.concat([df, pd.DataFrame([example])], ignore_index=True)
    
    # Write to jsonl file immediately after each example
    with open(output_file, 'a') as f:
        f.write(json.dumps(example) + '\n')
        f.flush() # Ensure it's written to disk
        
    if idx > 50:
       break


Processed prompts: 100%|██████████| 1/1 [00:13<00:00, 13.94s/it, est. speed input: 4.88 toks/s, output: 77.07 toks/s]
1it [00:13, 13.95s/it]

1.0


Processed prompts: 100%|██████████| 1/1 [02:54<00:00, 174.62s/it, est. speed input: 0.75 toks/s, output: 74.83 toks/s]
2it [03:08, 108.47s/it]

1.0


Processed prompts: 100%|██████████| 1/1 [00:27<00:00, 27.51s/it, est. speed input: 2.47 toks/s, output: 77.94 toks/s]
3it [03:36, 71.51s/it] 

...

1.0


Processed prompts: 100%|██████████| 1/1 [00:12<00:00, 12.70s/it, est. speed input: 4.80 toks/s, output: 78.99 toks/s]
6it [04:48, 35.60s/it]

1.0


Processed prompts: 100%|██████████| 1/1 [00:28<00:00, 28.89s/it, est. speed input: 1.38 toks/s, output: 78.57 toks/s]
7it [05:16, 33.41s/it]

1.0
